# 3.5 · 类别变量编码 / Categorical Encoding

> **课程定位 / Where this fits**
> 第 5 课，**Part 3 · EDA 与数据预处理**。
> Lesson 5, **Part 3 · EDA & Preprocessing**.
>
> 机器学习模型只吃**数字**，但真实数据里到处是文字类别：性别、城市、职业、教育程度……把它们正确地变成数字，是几乎每个项目都要做的一步。编码方式选错，要么浪费维度，要么**悄悄泄漏标签**。
> ML models only eat **numbers**, but real data is full of text categories: sex, city, occupation, education... Turning them into numbers correctly is a near-universal step. Pick the wrong encoding and you either waste dimensions or **silently leak the label**.
>
> 💼 **实战/面试视角**："类别特征怎么编码 / one-hot 的问题 / 高基数怎么办 / target 编码为什么会泄漏" 都是高频题。
> 💼 **Practical/interview angle:** "how to encode categoricals / one-hot's downsides / high cardinality / why target encoding leaks" are all common.

> 💡 **面试相关 / Interview-relevant**
> - "有序 vs 名义特征怎么编码"（出镜率 ★★★★★）
> - "One-Hot 的虚拟变量陷阱 / 维度爆炸"（★★★★）
> - "高基数类别(邮编/用户ID)怎么办"（★★★★★）
> - "目标编码为什么会泄漏 / 怎么防（K-fold + 平滑）"（★★★★★）
> - "测试集出现训练时没见过的类别怎么办"（★★★★）

---

## 学习目标 / Learning Objectives

1. 区分**有序(ordinal)**与**名义(nominal)**类别，分别用对的编码。
   Distinguish ordinal vs nominal categories and encode each correctly.
2. 掌握 **One-Hot**（含 drop='first' 与维度爆炸）。
   Master One-Hot (incl. drop='first' and the dimension blow-up).
3. 处理**高基数**：frequency / hashing / target 编码。
   Handle high cardinality: frequency / hashing / target encoding.
4. 看清**目标编码的泄漏陷阱**并用 K-fold + 平滑解决。
   See target encoding's leakage trap and fix it with K-fold + smoothing.
5. 处理测试集的**未见类别**。
   Handle unseen categories at test time.

## 目录 / TOC
1. [先建直觉 + 数据](#1)
2. [有序编码 Ordinal ⭐](#2)
3. [One-Hot 编码 ⭐](#3)
4. [高基数问题 + frequency/hashing ⭐](#4)
5. [目标编码的泄漏陷阱 ⭐](#5)
6. [K-fold 目标编码（防泄漏）⭐](#6)
7. [未见类别 + 小结](#7)


<a id="1"></a>
## 1. 先建直觉 + 数据 / Intuition & Data

编码前先问一个关键问题：**这个类别有没有内在顺序？**
Before encoding, ask the key question: **does this category have an inherent order?**
- **有序(ordinal)**：教育程度 高中 < 本科 < 硕士 < 博士。顺序有意义 → 编成 0,1,2,3 是对的。
  **Ordinal:** education HS < Bachelor < Master < PhD. Order is meaningful → encoding as 0,1,2,3 is right.
- **名义(nominal)**：国家、颜色、职业。没有顺序，**绝不能**随便编成 0,1,2——那会骗模型"日本 > 美国"。要用 One-Hot 之类不引入虚假顺序的编码。
  **Nominal:** country, color, occupation. No order, so you **must not** assign 0,1,2 — that fools the model into "Japan > USA". Use One-Hot or similar that introduces no fake order.

第三个维度是**基数(cardinality)**——这个类别有多少个不同取值。基数低（性别2类）随便编；基数高（邮编上万、用户ID上百万）就需要特殊招数。
A third axis is **cardinality** — how many distinct values. Low cardinality (sex: 2) is easy; high cardinality (zip codes in the thousands, user IDs in the millions) needs special tricks.

我们构造一个收入预测数据集，刚好覆盖这三种情况。
We build an income-prediction dataset covering all three cases.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

n = 5000
education = rng.choice(["HS","Bachelor","Master","PhD"], n, p=[0.4,0.35,0.2,0.05])  # 有序!
country = rng.choice(["US","UK","DE","JP","CA","MX","BR","IN"], n)                   # 名义, 低基数
occupation = rng.choice([f"job_{i}" for i in range(50)], n)                          # 名义, 高基数!

# 目标 high_income 受 education 影响(模拟真实信号) / target depends on education
edu_effect = pd.Series({"HS":0.1,"Bachelor":0.3,"Master":0.5,"PhD":0.7})
p_high = edu_effect[education].values + rng.normal(0, 0.1, n)            # 学历越高越可能高收入
high_income = (rng.random(n) < np.clip(p_high, 0, 1)).astype(int)       # 按概率采样标签

df = pd.DataFrame({"education":education,"country":country,"occupation":occupation,"high_income":high_income})
print(f"shape: {df.shape}")
print(f"education 基数 cardinality: {df.education.nunique()} (有序 ordinal)")
print(f"country    基数: {df.country.nunique()} (名义, 低基数 nominal low-card)")
print(f"occupation 基数: {df.occupation.nunique()} (名义, 高基数! high-card)")
print(f"目标 high_income 占比: {df.high_income.mean():.1%}")


<a id="2"></a>
## 2. 有序编码 Ordinal ⭐ / Ordinal Encoding

有序特征编成整数，但**必须手动指定顺序**。sklearn 的 `OrdinalEncoder` 默认按**字母序**编码——这会把"HS(高中)"排到"Bachelor"后面，顺序全乱。这是一个隐蔽但常见的 bug。
Encode an ordinal feature as integers, but **you must specify the order explicitly**. sklearn's `OrdinalEncoder` defaults to **alphabetical** order — which puts "HS" after "Bachelor", scrambling the order. A subtle but common bug.


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# 正确: 手动指定类别顺序(从低到高) / explicit order, low to high
edu_order = [["HS","Bachelor","Master","PhD"]]
oe = OrdinalEncoder(categories=edu_order)
df["edu_ordinal"] = oe.fit_transform(df[["education"]]).astype(int)
print("正确的有序编码(保序) correct ordinal encoding:")
print(df.groupby("education")["edu_ordinal"].first().sort_values())

# 错误: 默认按字母序 → 顺序乱掉 / WRONG: default alphabetical order scrambles it
oe_wrong = OrdinalEncoder()
oe_wrong.fit(df[["education"]])
print(f"\n默认字母序(错!) default alphabetical: {dict(zip(oe_wrong.categories_[0], range(4)))}")
print("→ HS(高中)=1 被排在 Bachelor=0 之后, 顺序全乱! 必须手动指定 categories")


<a id="3"></a>
## 3. One-Hot 编码 ⭐ / One-Hot Encoding

名义特征用 **One-Hot**：把 K 个类别变成 K 个 0/1 列，每行只有对应类别那一列是 1。这样不引入任何虚假顺序。
For nominal features use **One-Hot**: turn K categories into K 0/1 columns, with only the matching column = 1 per row. No fake order is introduced.

两个实战要点：
Two practical points:
- **虚拟变量陷阱(dummy variable trap)**：K 列里任一列都能由其余 K−1 列推出（完全共线）。**线性模型**会因此不稳定，所以用 `drop='first'` 去掉一列。树模型不受影响，无需 drop。
  **Dummy variable trap:** any one of the K columns is determined by the other K−1 (perfect collinearity). This destabilizes **linear models**, so use `drop='first'`. Trees don't care.
- **维度爆炸**：K 大时 One-Hot 列数爆炸（邮编→上万列），需改用第 4 节的方法。
  **Dimension blow-up:** large K explodes the column count (zip→thousands), needing Section 4's methods.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

# country(8类) → one-hot 8列 / nominal -> one-hot
ohe = OneHotEncoder(sparse_output=False)
country_oh = ohe.fit_transform(df[["country"]])
print(f"country(8类) → one-hot: {country_oh.shape[1]} 列, 列名 {list(ohe.get_feature_names_out())}")

# drop='first' 去掉一列避免虚拟变量陷阱(给线性模型用) / drop one for linear models
ohe_drop = OneHotEncoder(sparse_output=False, drop="first")
print(f"drop='first' → {ohe_drop.fit_transform(df[['country']]).shape[1]} 列 (k-1, 防共线, 线性模型用)")
print(f"pandas 快捷: pd.get_dummies(df['country']) → {pd.get_dummies(df['country']).shape[1]} 列")


<a id="4"></a>
## 4. 高基数问题 + frequency/hashing ⭐ / High Cardinality

`occupation` 有 50 类，One-Hot 还能忍。但想象**邮编→4 万列、用户ID→百万列**——One-Hot 直接崩溃，也喂不进模型。高基数的几种解法：
`occupation` has 50 categories — One-Hot is still OK. But imagine **zip → 40k columns, user ID → millions** — One-Hot collapses. Solutions for high cardinality:
- **Frequency 编码**：用该类别的**出现频率**代替它（50 类→1 列）。简单有效。
  **Frequency encoding:** replace a category with its **frequency** (50 cats → 1 column). Simple and effective.
- **Hashing 编码**：用哈希函数把任意基数压到**固定列数**。维度可控、对未见类别免疫；缺点是**碰撞**（不同类别可能撞到同一列）、不可解释。
  **Hashing:** hash to a **fixed number of columns**. Controllable dimension, immune to unseen categories; downside is **collisions** and no interpretability.
- **Target 编码**（第 5–6 节）：用该类别的目标均值代替（1 列，最强但有泄漏陷阱）。
  **Target encoding** (Sections 5–6): replace with the category's target mean (1 column, strongest but has a leakage trap).
- **Embedding**：神经网络学低维稠密向量（Part 12/推荐系统标配）。
  **Embedding:** neural nets learn dense low-dim vectors (standard in deep/recsys).


In [ ]:
# Frequency 编码: 类别 → 它的出现频率 / category -> its frequency
freq_map = df["occupation"].value_counts(normalize=True)    # 每个类别的占比
df["occ_freq"] = df["occupation"].map(freq_map)             # 用 map 把类别替换成频率值
print("Frequency 编码: 50 个类别 → 1 列, 无维度爆炸")
print(df[["occupation","occ_freq"]].drop_duplicates().head(4).round(4).to_string(index=False))

# Hashing 编码: 把任意基数哈希到固定列数 / hash to a fixed number of columns
from sklearn.feature_extraction import FeatureHasher
hasher = FeatureHasher(n_features=8, input_type="string")
hashed = hasher.transform([[v] for v in df["occupation"]]).toarray()
print(f"\nHashing 编码: occupation(50类) → 固定 {hashed.shape[1]} 列")
print("优点: 维度可控 + 对未见类别免疫; 缺点: 碰撞(不同类别撞同列) + 不可解释")


<a id="5"></a>
## 5. 目标编码的泄漏陷阱 ⭐ / Target Encoding's Leakage Trap

**目标编码(target encoding)**：用"该类别下目标的平均值"代替类别（如某职业的平均高收入率）。它能把高基数压成 1 列，且直接编码了"这个类别和目标的关系"，威力很大。**但朴素做法会泄漏标签**——这是面试高频陷阱。
**Target encoding:** replace a category with "the target's mean within that category" (e.g. a job's average high-income rate). It compresses high cardinality to 1 column and directly encodes the category↔target relationship — very powerful. **But the naive version leaks the label** — a frequently-tested trap.

问题在于：给某一行编码它的职业时，用到的"该职业均值"里**包含了这一行自己的 y**。对**稀有类别**（只出现几次）尤其致命——编码值几乎就是它自己标签的复制。结果：训练集上看起来完美，测试集上崩盘。
The problem: when encoding a row's job, the "job mean" used **includes that row's own y**. This is fatal for **rare categories** (appearing a few times) — the encoding is nearly a copy of its own label. Result: looks perfect on train, collapses on test.


In [ ]:
# ❌ 朴素 target 编码: 用全部数据(含每行自己)算类别均值 → 泄漏 / naive target encoding LEAKS
naive = df.groupby("occupation")["high_income"].mean()
df["occ_target_naive"] = df["occupation"].map(naive)
print("朴素 target 编码: 编码某行时, 类别均值里包含了'这一行自己的 y' → 标签泄漏进特征")

# 演示泄漏严重性: 取最稀有的类别, 它的编码几乎=自己那几行的 y / show leakage on rarest category
rare_cat = df["occupation"].value_counts().index[-1]
sub = df[df.occupation == rare_cat]
print(f"\n最稀有类别 '{rare_cat}' 只出现 {len(sub)} 次")
print(f"它的朴素 target 编码 = {naive[rare_cat]:.2f}, 几乎直接复制了这几行的 y(={list(sub.high_income)})")


<a id="6"></a>
## 6. K-fold 目标编码（防泄漏）⭐ / Leak-free Target Encoding

正确做法两步：
The fix has two parts:
1. **K-fold（out-of-fold）**：把数据分成 K 折，编码某一折时**只用其他折**算均值——这样一行的编码绝不含它自己的 y。
   **K-fold (out-of-fold):** split into K folds; encode a fold using **only the other folds** to compute means — so a row's encoding never includes its own y.
2. **平滑(smoothing)**：稀有类别的均值不可靠，让它**向全局均值收缩**（这正是 2.10 的贝叶斯收缩思想：证据少就更信先验）。
   **Smoothing:** a rare category's mean is unreliable, so **shrink it toward the global mean** (the Bayesian shrinkage idea from 2.10: with little evidence, trust the prior more).


In [ ]:
from sklearn.model_selection import KFold

def kfold_target_encode(df, col, target, n_splits=5, smoothing=10, seed=0):
    global_mean = df[target].mean()                    # 全局目标均值(收缩的方向)
    encoded = np.zeros(len(df))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(df):
        tr, val = df.iloc[tr_idx], df.iloc[val_idx]    # 用 tr 折算均值, 编码 val 折(val 不参与自己均值)
        stats = tr.groupby(col)[target].agg(["mean","count"])
        # 平滑公式: count 大→信类别均值, count 小→向全局均值收缩(贝叶斯, 2.10)
        smooth = (stats["mean"]*stats["count"] + global_mean*smoothing) / (stats["count"]+smoothing)
        encoded[val_idx] = val[col].map(smooth).fillna(global_mean).values  # 未见类别→全局均值
    return encoded

df["occ_target_kfold"] = kfold_target_encode(df, "occupation", "high_income")
rare_cat = df["occupation"].value_counts().index[-1]
print("K-fold target 编码: 每行编码只用'其他折'数据 → 不含自己 → 无泄漏")
print(f"\n同一稀有类别对比:")
print(f"  朴素(泄漏)  naive:  {df[df.occupation==rare_cat]['occ_target_naive'].iloc[0]:.3f}")
print(f"  K-fold(安全) safe:   {df[df.occupation==rare_cat]['occ_target_kfold'].mean():.3f} (向全局均值 {df.high_income.mean():.3f} 收缩)")
print("\n💡 sklearn 1.3+ 内置 TargetEncoder, 自动 K-fold + 平滑, 实战优先用它")


<a id="7"></a>
## 7. 未见类别 + 小结 / Unseen Categories & Summary

**生产里必踩的坑**：训练时只见过某些类别，上线后来了**新类别**（如新国家、新职业）。One-Hot 默认会**直接报错**。必须设 `handle_unknown='ignore'`，让未见类别编码成全 0（或对 target/frequency 编码用全局均值兜底）。
**A production gotcha:** training saw only some categories, but live traffic brings **new ones** (a new country, a new job). One-Hot **errors out** by default. Set `handle_unknown='ignore'` so unseen categories become all-zeros (or fall back to the global mean for target/frequency encoding).


In [ ]:
from sklearn.preprocessing import OneHotEncoder

train_country = pd.DataFrame({"country": ["US","UK","DE","JP"]*10})
test_country  = pd.DataFrame({"country": ["US","BR_NEW","DE"]})   # BR_NEW 训练时没见过

# handle_unknown='ignore': 未见类别编码成全 0, 不报错 / unseen -> all-zeros, no crash
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(train_country)
test_oh = ohe.transform(test_country)
print(f"train 见过的类别 known: {list(ohe.categories_[0])}")
print(f"test 'BR_NEW'(未见) 的 one-hot 行: {test_oh[1]} (全 0, 不报错)")
print("⚠ 默认 handle_unknown='error' 会在 test 直接崩溃 → 生产代码必须设 'ignore'")


```
先分类: 有序(ordinal, 指定顺序!) vs 名义(nominal, 别编整数) + 看基数高低
有序: OrdinalEncoder(categories=...) — 默认字母序是坑
名义低基数: One-Hot; 线性模型 drop='first' 防虚拟变量陷阱; 树不用 drop
高基数(邮编/ID): frequency / hashing(固定列, 有碰撞) / target / embedding
目标编码: 用类别目标均值(强但泄漏); 朴素版含自己的 y → 稀有类别灾难
  防泄漏: K-fold(out-of-fold) + 平滑(向全局均值收缩, 贝叶斯 2.10); 用 sklearn TargetEncoder
未见类别: One-Hot 设 handle_unknown='ignore'; 生产必备
```

### 💡 面试速查 / Interview cheat-sheet
1. **有序→指定顺序的 Ordinal；名义→One-Hot**（别给名义编整数）。
   Ordinal → ordered OrdinalEncoder; nominal → One-Hot (never integer-code nominals).
2. **One-Hot drop='first'** 防虚拟变量陷阱（线性模型）；树不用。
   drop='first' avoids the dummy trap (linear models); trees don't need it.
3. **高基数**：frequency / hashing(碰撞) / target / embedding。
   High cardinality: frequency / hashing(collisions) / target / embedding.
4. **目标编码泄漏** → K-fold + 平滑（贝叶斯收缩）；用 sklearn TargetEncoder。
   Target encoding leaks → K-fold + smoothing; use sklearn's TargetEncoder.
5. **未见类别** → handle_unknown='ignore'（生产必备）。
   Unseen categories → handle_unknown='ignore' (production must-have).

### 下一节 / Next
**3.6 特征工程**——编码之后，怎么"造"出更有预测力的新特征：交互、分箱、时间/周期特征、聚合特征等。
**3.6 Feature Engineering** — after encoding, how to *create* more predictive features: interactions, binning, datetime/cyclical features, aggregations.
